In [0]:
def pn_api_extraction(api_url,api_param,headers,offset,limit):

    all_barca_players = []    
    
    while True:
        api_param['limit'] = limit
        api_param['offset'] = offset

        res = requests.get(api_url, params=api_param, headers=headers)
        data = res.json()
        players = data.get("player_rankings", [])

        # Stop when no more data
        if not players:
            break

        for p in players:
            if not isinstance(p, dict):
                continue

            # Filter FC Barcelona players
            if p.get("team", {}).get("slug") == "fc-barcelona":
                all_barca_players.append(p)

        offset += limit
    
    json_data = json.dumps(all_barca_players, indent=4, ensure_ascii=False)
    
    return json_data

def api_extraction(api_url,api_param):

    all_barca_players = []

    response = requests.get(api_url, params=api_param)

    #  json_data = json.loads(response.text)
    json_data = json.dumps(response.json(), indent=4, ensure_ascii=False)
    
    return json_data

In [0]:
def upload_s3_vol(json_data,bucket_name,s3_key,volume_path):
    s3_error_message = {}
    dbk_error_message = {}

    try:
        # Create S3 client
        s3 = boto3.client(
            "s3",
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key,
            region_name="us-east-1"
        )

        # Upload to S3
        s3.put_object(
            Bucket=bucket_name,
            Key=s3_key,
            Body=json_data,
            ContentType="application/json"
        )
        print(f"File uploaded to S3 the path:{s3_key}")
    except Exception as e:
        s3_error_message[file_name] = f"Error uploading file to S3: {str(e)}"
       
    try:
        response = requests.put(
            f"{databricks_host}/api/2.0/fs/files{volume_path}",
            headers={
                "Authorization": f"Bearer {databricks_token}",
                "Content-Type": "application/octet-stream"
            },
            data=json_data.encode("utf-8")
        )

        print(f"File uploaded to volume {volume_path}")
    except Exception as e:
        dbk_error_message[file_name] = f"Error uploading file to volume: {str(e)}"
    
    # if s3_error_message[file_name] or dbk_error_message[file_name]:
    #     print(f"Error messages for {file_name}:")
    #     print(s3_error_message[file_name])
    #     print(dbk_error_message[file_name])